# Phase 5d: Cross-stream synthesis

Joins Stream A (RAGAS), Stream B (LLM judge), and Stream C (drift) per question.

**Outputs:** `data/06_evaluation/stream_d/v14_cross_stream_synthesis/joined_per_question.csv`, `significance_test.csv`, and `figures/*.png`.

#Composer through a Cursor IDE guided the development of this script

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 140


In [ ]:
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "scripts" else Path.cwd().resolve()
SCRIPT_DIR = BASE_DIR / "scripts"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
from pipeline_constants import INFERENCE_TAG, RAGAS_EVAL_TAG, LLM_JUDGE_EVAL_TAG

PROMPT_VERSION = INFERENCE_TAG
RAGAS_TAG = RAGAS_EVAL_TAG
JUDGE_TAG = LLM_JUDGE_EVAL_TAG
DRIFT_TAG = INFERENCE_TAG
OUT_TAG = f"{INFERENCE_TAG}_cross_stream_synthesis"
OUT_DIR = BASE_DIR / "data" / "06_evaluation" / "stream_d" / OUT_TAG
FIG_DIR = OUT_DIR / "figures"
JOINED_CSV = OUT_DIR / "joined_per_question.csv"
SIGNIF_CSV = OUT_DIR / "significance_test.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert JOINED_CSV.exists(), f"Missing joined table: {JOINED_CSV}"
print(f"OUT_DIR={OUT_DIR}")
print(f"JOINED_CSV={JOINED_CSV.name}")


## PostgreSQL join 

Shows the SQL used when scores were synced to `send_rag_db`.


In [ ]:
# Evidence only —
# Original join: responses + questions + ragas_scores + judge_scores + drift_scores.

SQL_ARCHIVE = '''
WITH base AS (
    SELECT
        r.question_id, q.category, rs.faithfulness, rs.answer_relevancy,
        js.tone_alignment_score as tone, js.accuracy_score as accuracy, js.grounding_score as grounding,
        ds.sim_answer_context as sim_ctx
    FROM responses r
    JOIN questions q ON q.id = r.question_id
    LEFT JOIN ragas_scores rs ON rs.response_id = r.id AND rs.eval_tag = %(ragas_tag)s
    LEFT JOIN judge_scores js ON js.response_id = r.id AND js.eval_tag = %(judge_tag)s
    LEFT JOIN drift_scores ds ON ds.response_id = r.id AND ds.eval_tag = %(drift_tag)s
    WHERE r.dataset = 'baseline' AND r.prompt_version = %(prompt_version)s
),
hyb AS (
    SELECT
        r.question_id, rs.faithfulness, rs.answer_relevancy,
        js.tone_alignment_score as tone, js.accuracy_score as accuracy, js.grounding_score as grounding,
        ds.sim_answer_context as sim_ctx
    FROM responses r
    LEFT JOIN ragas_scores rs ON rs.response_id = r.id AND rs.eval_tag = %(ragas_tag)s
    LEFT JOIN judge_scores js ON js.response_id = r.id AND js.eval_tag = %(judge_tag)s
    LEFT JOIN drift_scores ds ON ds.response_id = r.id AND ds.eval_tag = %(drift_tag)s
    WHERE r.dataset = 'hybrid_rag' AND r.prompt_version = %(prompt_version)s
)
SELECT
    b.question_id, b.category,
    b.faithfulness AS b_faith, h.faithfulness AS h_faith,
    b.grounding AS b_ground, h.grounding AS h_ground,
    b.accuracy AS b_acc, h.accuracy AS h_acc,
    b.tone AS b_tone, h.tone AS h_tone,
    b.answer_relevancy AS b_rel, h.answer_relevancy AS h_rel,
    b.sim_ctx AS b_sim_ctx, h.sim_ctx AS h_sim_ctx
FROM base b
JOIN hyb h ON h.question_id = b.question_id
'''

print("SQL query retained for methodology evidence")


## 1. Load joined per-question table


In [ ]:
df = pd.read_csv(JOINED_CSV)

metrics = ["faith", "ground", "acc", "tone", "rel", "sim_ctx"]
required = ["question_id", "category", "improved_core"] + [f"delta_{m}" for m in metrics]
missing = [c for c in required if c not in df.columns]
if missing:
    for m in metrics:
        b_col, h_col = f"b_{m}", f"h_{m}"
        if f"delta_{m}" not in df.columns and b_col in df.columns and h_col in df.columns:
            df[f"delta_{m}"] = df[h_col] - df[b_col]
    if "improved_core" not in df.columns:
        df["improved_core"] = ((df["delta_faith"] > 0) | (df["delta_ground"] > 0)).astype(int)
    still_missing = [c for c in required if c not in df.columns]
    if still_missing:
        raise ValueError(f"joined_per_question.csv missing columns: {still_missing}")

print(f"Loaded {len(df)} rows from {JOINED_CSV.name}")
df.head()


## 2. Significance tests


In [ ]:
stats_rows = []
for m in metrics:
    vals = df[f"delta_{m}"].dropna()
    _, p = wilcoxon(vals)
    stats_rows.append({
        "metric": m,
        "mean_delta": float(vals.mean()),
        "p_value": float(p),
        "sig": bool(p < 0.05),
    })

significance_df = pd.DataFrame(stats_rows)
significance_df.to_csv(SIGNIF_CSV, index=False)
print(f"Saved {SIGNIF_CSV.name}")
significance_df


## 3. Figures


In [ ]:
plt.figure(figsize=(10, 8))
delta_cols = [f"delta_{m}" for m in metrics]
sns.heatmap(df[delta_cols].corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Heatmap: Interplay Between Technical Grounding and Pedagogical Tone")
plt.tight_layout()
plt.savefig(FIG_DIR / "metric_synergy_heatmap.png", dpi=300, bbox_inches="tight")
plt.close()


In [ ]:
base_features = [f"b_{m}" for m in metrics]
X = df[base_features].copy()
X = pd.concat([X, pd.get_dummies(df["category"], prefix="cat")], axis=1)
y = df["improved_core"]

X = X.dropna()
y = y.loc[X.index]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(max_iter=1000)
model.fit(X_scaled, y)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": np.abs(model.coef_[0]),
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=importance_df, x="Importance", y="Feature", color="steelblue")
plt.title("Drivers of Success: Top Predictors of Hybrid-RAG Uplift")
plt.xlabel("Importance (Standardized Coefficient)")
x_max = importance_df["Importance"].max()
x_pad = x_max * 0.03
for patch in ax.patches:
    width = patch.get_width()
    y_pos = patch.get_y() + patch.get_height() / 2
    ax.text(width + x_pad, y_pos, f"{width:.2f}", ha="left", va="center", fontsize=9)
ax.set_xlim(0, x_max * 1.15)
plt.tight_layout()
plt.savefig(FIG_DIR / "predictor_importance_uplift.png", dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved {FIG_DIR / 'predictor_importance_uplift.png'}")


In [ ]:
uplift = df.groupby("category")["improved_core"].mean().reset_index()
uplift["uplift_pct"] = uplift["improved_core"] * 100

plt.figure(figsize=(9, 5))
ax = sns.barplot(data=uplift, x="category", y="uplift_pct", hue="category", palette="magma", legend=False)
plt.title("Hybrid-RAG Success Rate by SEND Domain")
plt.ylabel("% Responses with Improved Grounding/Faithfulness")
plt.xlabel("")
plt.xticks(rotation=45, ha="right")
for patch in ax.patches:
    height = patch.get_height()
    if height <= 0:
        continue
    ax.text(
        patch.get_x() + patch.get_width() / 2,
        height / 2,
        f"{height:.1f}%",
        ha="center",
        va="center",
        color="white" if height >= 40 else "black",
        fontsize=10,
        fontweight="bold",
    )
plt.tight_layout()
plt.savefig(FIG_DIR / "category_safety_uplift.png", dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved {FIG_DIR / 'category_safety_uplift.png'}")
